<a href="https://colab.research.google.com/github/Shaithra2107/HALT-RAG/blob/lift_signal/HALT_RAG_v5_signals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HALT-RAG v5 — Signals, SWL, Ablation

**What changed from v4, and why.**

| # | v4 did | v5 does | Why |
|---|--------|---------|-----|
| 1 | Lift's 50 "random" docs were sampled from the 1500 passages that were top-5 for *some* question | samples from the whole corpus | those 1500 were all good passages, so the baseline was far too strong and Lift came out too low |
| 2 | clarity's background counted only the retrieved passages | counts the whole corpus | the real clarity score (Cronen-Townsend 2002) is a divergence from the *collection* |
| 3 | CAS = softmax probability at `[CLS]` | standard SQuAD 2.0 null score: `null_logits - best_span_logits` | the softmax proxy ignores how good the best actual answer span is |
| 4 | 500 convenience distractors | `n_distractors` raised | with 620 pages total, "the corpus" was a toy |
| 5 | reported on all 300 | reports on all 300 **and** the 136 with known coverage | 164 questions never had their gold page reached by the scan, so their coverage label is "we didn't look there" |
| 6 | — | validates CAS against your 50 hand-labelled rows | the only human ground truth you have |
| 7 | string-match baseline vs `y_cov` | skipped when they are the same column | it would score exactly 1.000 and mean nothing |

Run order is top to bottom. Cells 2–4 need the KILT cache in Drive.

## Cell 1 — Install

In [ ]:
!pip install -q sentence-transformers==3.0.1 2>&1 | tail -2
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
torch 2.11.0+cpu | cuda: False


## Cell 2 — Drive + CONFIG

In [ ]:
import os, json
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/halt_rag_cache'
os.makedirs(DRIVE_DIR, exist_ok=True)

CONFIG = {
    # prefers the v5 scan's output; falls back to the old file if you have not run it yet
    'audit_csv'    : (f'{DRIVE_DIR}/halt_rag_audit_v5.csv'
                     if os.path.exists(f'{DRIVE_DIR}/halt_rag_audit_v5.csv')
                     else f'{DRIVE_DIR}/halt_rag_audit_full.csv'),
    'labelled_csv' : f'{DRIVE_DIR}/halt_rag_audit_prelabeled.csv',  # your 50 hand labels
    'out_signals'  : f'{DRIVE_DIR}/halt_rag_v5_signals.csv',
    'out_results'  : f'{DRIVE_DIR}/halt_rag_v5_results.json',

    'bi_encoder'   : 'sentence-transformers/all-MiniLM-L6-v2',
    'qa_model'     : 'deepset/roberta-base-squad2',
    'lm'           : 'google/flan-t5-base',

    'top_k'        : 5,
    'chunk_words'  : 100,

    # FIX 4 — the corpus has to be big enough to be a corpus
    'n_distractors': 5000,     # was 500. raise as far as RAM allows
    'kilt_scan_limit': 3_000_000,

    # FIX 1 — Lift's baseline is drawn from this many random CORPUS passages
    'lift_pool'    : 5000,

    'max_ans_len'  : 30,       # CAS: cap on answer span length

    # load the cached corpus embeddings from Drive instead of re-encoding.
    # Set False only if you want to force a fresh encode.
    'reuse_embeddings': True,

    'seed'         : 42,
}
print(json.dumps(CONFIG, indent=2))
assert os.path.exists(CONFIG['audit_csv']), f"Put halt_rag_audit_full.csv in {DRIVE_DIR}/"

Mounted at /content/drive
{
  "audit_csv": "/content/drive/MyDrive/halt_rag_cache/halt_rag_audit_v5.csv",
  "labelled_csv": "/content/drive/MyDrive/halt_rag_cache/halt_rag_audit_prelabeled.csv",
  "out_signals": "/content/drive/MyDrive/halt_rag_cache/halt_rag_v5_signals.csv",
  "out_results": "/content/drive/MyDrive/halt_rag_cache/halt_rag_v5_results.json",
  "bi_encoder": "sentence-transformers/all-MiniLM-L6-v2",
  "qa_model": "deepset/roberta-base-squad2",
  "lm": "google/flan-t5-base",
  "top_k": 5,
  "chunk_words": 100,
  "n_distractors": 5000,
  "kilt_scan_limit": 3000000,
  "lift_pool": 5000,
  "max_ans_len": 30,
  "reuse_embeddings": true,
  "seed": 42
}


## Cell 3 — Load the KILT cache and rebuild the corpus

Same cache the validation notebook wrote, so no 3M re-scan. The one change is
that we keep **all** the distractor pages the cache holds, instead of 500.

If the cache was built with `n_distractors = 500` you are capped at 500 here —
in that case re-run the validation notebook's scan cell with a larger
`n_distractors` first. The printout below tells you which case you are in.

In [ ]:
import pickle, hashlib, re
import pandas as pd, numpy as np

def normalise_title(t): return re.sub(r'_', ' ', str(t).lower().strip())

audit = pd.read_csv(CONFIG['audit_csv'])
print(f"audit CSV : {os.path.basename(CONFIG['audit_csv'])}  ({len(audit)} rows)")

# prefer the v5 scan's cache (full pass + reservoir sample). Fall back to the old
# hashed cache, which holds only 500 first-in-stream distractors.
V5_CACHE = os.path.join(DRIVE_DIR, 'kilt_cache_v5.pkl')
if os.path.exists(V5_CACHE):
    KILT_CACHE = V5_CACHE
    print("cache     : kilt_cache_v5.pkl  (full pass, reservoir sample)")
else:
    targets = {normalise_title(t) for t in audit['wiki_title']}
    _sig = hashlib.md5(
        (str(CONFIG['kilt_scan_limit']) + '|' + '|'.join(sorted(targets))).encode()
    ).hexdigest()[:12]
    KILT_CACHE = os.path.join(DRIVE_DIR, f"kilt_cache_{_sig}.pkl")
    print(f"cache     : {os.path.basename(KILT_CACHE)}  (OLD partial scan)")
    print("\n!! You have not run HALT_RAG_v5_scan.ipynb yet.")
    print("   Lift's baseline will come from 500 first-in-stream pages, which is")
    print("   not a sample of the collection, so Lift will not mean what it should.")
    print("   Run the scan notebook first unless you are only testing the plumbing.")
assert os.path.exists(KILT_CACHE), f"cache not found: {KILT_CACHE}"

with open(KILT_CACHE, 'rb') as f:
    _c = pickle.load(f)
kilt_pages      = _c['kilt_pages']
distractor_pool = _c['distractor_pool']
print(f"entity pages in cache : {len(kilt_pages)}")
print(f"distractors in cache  : {len(distractor_pool)}")
if len(distractor_pool) < CONFIG['n_distractors']:
    print(f"\n!! cache holds only {len(distractor_pool)} distractors, you asked for "
          f"{CONFIG['n_distractors']}.\n"
          f"   Lift's baseline will be weak. Re-run the validation scan with a bigger\n"
          f"   n_distractors to fix this properly.")

def chunk_text(paragraphs, w=CONFIG['chunk_words']):
    out = []
    for para in paragraphs:
        words = str(para).split()
        for i in range(0, len(words), w):
            c = ' '.join(words[i:i+w])
            if len(c.strip()) > 30: out.append(c)
    return out

passages, titles, is_entity = [], [], []
for norm_title, paras in kilt_pages.items():
    for c in chunk_text(paras):
        passages.append(c); titles.append(norm_title); is_entity.append(1)
for d in distractor_pool[:CONFIG['n_distractors']]:
    for c in chunk_text(d['paragraphs']):
        passages.append(c); titles.append(d['title']); is_entity.append(0)

is_entity = np.array(is_entity)
print(f"\ncorpus: {len(passages):,} passages "
      f"({is_entity.sum():,} from entity pages, {(1-is_entity).sum():,} distractor)")
print("NOTE: this is still a pilot corpus, not full KILT. State that in the write-up.")

audit CSV : halt_rag_audit_v5.csv  (300 rows)
cache     : kilt_cache_v5.pkl  (full pass, reservoir sample)
entity pages in cache : 133
distractors in cache  : 10000

corpus: 50,035 passages (2,835 from entity pages, 47,200 distractor)
NOTE: this is still a pilot corpus, not full KILT. State that in the write-up.


## Cell 4 — Embed the corpus, build the index, retrieve top-5

The embeddings are cached to Drive as a `.npy` file, so this only costs you the
full encoding time **once**. On later runs it loads in seconds.

The cache is keyed on the number of passages. If you re-scan with a different
reservoir size the count changes, the cache is treated as stale, and it
re-encodes automatically. Set `reuse_embeddings` to False to force that by hand.

In [ ]:
!pip install -q faiss-cpu

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
bi = SentenceTransformer(CONFIG['bi_encoder'], device=DEV)

EMB_NPY  = f"{DRIVE_DIR}/corp_embs_v5.npy"
EMB_META = f"{DRIVE_DIR}/corp_embs_v5.meta.json"

def _meta_ok():
    if not (os.path.exists(EMB_NPY) and os.path.exists(EMB_META)): return False
    try:
        m = json.load(open(EMB_META))
    except Exception:
        return False
    return (m.get('n_passages') == len(passages)
            and m.get('chunk_words') == CONFIG['chunk_words']
            and m.get('bi_encoder') == CONFIG['bi_encoder'])

if CONFIG['reuse_embeddings'] and _meta_ok():
    corp_embs = np.load(EMB_NPY)
    print(f"loaded cached embeddings: {corp_embs.shape}  ({EMB_NPY})")
else:
    if os.path.exists(EMB_NPY):
        print("cached embeddings are stale (corpus or model changed) — re-encoding")
    print(f"encoding {len(passages):,} passages ... this is the one slow part")
    corp_embs = bi.encode(passages, batch_size=256, show_progress_bar=True,
                          normalize_embeddings=True).astype('float32')
    np.save(EMB_NPY, corp_embs)
    json.dump({'n_passages': len(passages),
               'chunk_words': CONFIG['chunk_words'],
               'bi_encoder': CONFIG['bi_encoder'],
               'dim': int(corp_embs.shape[1])}, open(EMB_META, 'w'))
    mb = corp_embs.nbytes / 1e6
    print(f"saved {EMB_NPY}  ({mb:,.0f} MB) — later runs will load this instead")

assert len(corp_embs) == len(passages), \
    f"embeddings ({len(corp_embs)}) do not match passages ({len(passages)})"

index = faiss.IndexFlatIP(corp_embs.shape[1])
index.add(corp_embs)
print(f"index: {index.ntotal:,} vectors, dim={corp_embs.shape[1]}")

# question embeddings are tiny (300 rows), never worth caching
q_emb = bi.encode(audit['question'].tolist(), batch_size=128,
                  show_progress_bar=True, normalize_embeddings=True).astype('float32')
ret_scores, ret_idx = index.search(q_emb, CONFIG['top_k'])
print("retrieved:", ret_idx.shape)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (7 outputs left out)

Batches:   0%|          | 0/196 [00:00<?, ?it/s]

saved /content/drive/MyDrive/halt_rag_cache/corp_embs_v5.npy  (77 MB) — later runs will load this instead
index: 50,035 vectors, dim=384


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

retrieved: (300, 5)


## Cell 5 — Signals 1–3: Lift (fixed), relevance, consistency

**The fix.** `LIFT_POOL` is a random sample of the corpus, taken once. The old
code sampled from the retrieved passages, which are all passages a retriever
already liked — so the bar was set far too high.

Any pool passage that happens to be in a question's own top-5 is dropped for
that question, so Lift never compares a retrieval against itself.

In [ ]:
rng = np.random.default_rng(CONFIG['seed'])

# FIX 1 — one random sample of the CORPUS, used as the baseline for every question
pool_sz   = min(CONFIG['lift_pool'], len(corp_embs))
pool_idx  = rng.choice(len(corp_embs), size=pool_sz, replace=False)
LIFT_POOL = corp_embs[pool_idx]
pool_set  = {int(j): p for p, j in enumerate(pool_idx)}   # corpus idx -> pool row
print(f"Lift baseline pool: {pool_sz:,} random corpus passages")

lift, relevance, consistency, rnd_mu, rnd_sd = [], [], [], [], []
for i in range(len(audit)):
    qv   = q_emb[i]
    pv   = corp_embs[ret_idx[i]]          # (5, d) the retrieved passages
    sims = pv @ qv
    relevance.append(float(sims.mean()))

    G  = pv @ pv.T
    iu = np.triu_indices(len(pv), k=1)
    consistency.append(float(G[iu].mean()))

    # drop this question's own passages from the baseline, if present
    drop = [pool_set[int(j)] for j in ret_idx[i] if int(j) in pool_set]
    rs   = LIFT_POOL @ qv
    if drop:
        rs = np.delete(rs, drop)

    mu, sd = float(rs.mean()), float(rs.std())
    rnd_mu.append(mu); rnd_sd.append(sd)
    lift.append((float(sims.mean()) - mu) / sd if sd > 0 else 0.0)

df = audit.copy()
df['lift']        = lift
df['relevance']   = relevance
df['consistency'] = consistency
df['rand_mu']     = rnd_mu
df['rand_sd']     = rnd_sd
df['ret_idx']     = [list(map(int, r)) for r in ret_idx]
df['ret_text']    = [[passages[int(j)] for j in r] for r in ret_idx]

print(df[['lift','relevance','consistency','rand_mu','rand_sd']].describe().round(3).to_string())
print("\nSanity: rand_mu should now be LOW (random corpus docs are unlike the question).")
print("If rand_mu is close to relevance, the corpus is too small or too narrow.")

Lift baseline pool: 5,000 random corpus passages
          lift  relevance  consistency  rand_mu  rand_sd
count  300.000    300.000      300.000  300.000  300.000
mean     5.052      0.521        0.413    0.065    0.091
std      0.825      0.064        0.120    0.016    0.006
min      3.656      0.377        0.205    0.022    0.070
25%      4.453      0.476        0.327    0.054    0.087
50%      4.839      0.511        0.388    0.065    0.091
75%      5.457      0.560        0.468    0.077    0.095
max      7.582      0.707        1.000    0.104    0.122

Sanity: rand_mu should now be LOW (random corpus docs are unlike the question).
If rand_mu is close to relevance, the corpus is too small or too narrow.


## Cell 6 — Signals 4–5: kw_coverage, clarity (fixed)

**The fix.** `corpus_counter` now counts every passage in the corpus. The old
code counted only the retrieved ones, which turned clarity into "how unusual is
this entity compared to the other 299 entities" — close to a definition of
obscurity, which is exactly how it behaved.

In [ ]:
import math
from collections import Counter

STOP = set('''a an the is are was were be been being of in on at to for with by from as and or but
if then than that this these those what which who whom whose when where why how do does did done
it its his her their there here not no nor so such own same too very can will just should now
you your i me my we our they them he she him'''.split())

def toks(s):   return re.findall(r"[a-z0-9]+", str(s).lower())
def content(w): return [x for x in w if x not in STOP and len(x) > 2]

# FIX 2 — background language model over the WHOLE corpus
print("building corpus language model ...")
corpus_counter = Counter()
for t in passages:
    corpus_counter.update(content(toks(t)))
TOT = sum(corpus_counter.values())
print(f"corpus LM: {len(corpus_counter):,} unique terms, {TOT:,} tokens")

def kw_coverage(q, ps):
    kws = set(content(toks(q)))
    if not kws: return 0.0
    return len(kws & set(toks(' '.join(ps)))) / len(kws)

def clarity(ps, smooth=1e-9):
    c = Counter(content(toks(' '.join(ps)))); n = sum(c.values())
    if n == 0: return 0.0
    return float(sum((f/n) * math.log((f/n) / (corpus_counter.get(w, 0)/TOT or smooth))
                     for w, f in c.items()))

df['kw_coverage'] = [kw_coverage(q, ps) for q, ps in zip(df['question'], df['ret_text'])]
df['clarity']     = [clarity(ps) for ps in df['ret_text']]
print(df[['kw_coverage','clarity']].describe().round(3).to_string())

building corpus language model ...
corpus LM: 94,667 unique terms, 1,277,790 tokens
       kw_coverage  clarity
count      300.000  300.000
mean         0.566    5.041
std          0.311    0.592
min          0.000    3.673
25%          0.333    4.656
50%          0.500    4.994
75%          0.750    5.384
max          1.000    7.289


## Cell 7 — Signal 6: CAS (fixed)

**The fix.** The standard SQuAD 2.0 decision compares two numbers:

* `null  = start_logit[CLS] + end_logit[CLS]` — the model's "no answer here" score
* `best  = max over valid spans of (start_logit[a] + end_logit[b])`

`score_diff = null - best`. High means unanswerable. Answerability is `-score_diff`.

v4 instead read the softmax probability at `[CLS]`, which never looks at how
good the best actual span was. Both are computed here so you can compare them,
and the comparison is worth a paragraph in the dissertation.

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from tqdm.auto import tqdm

qa_tok = AutoTokenizer.from_pretrained(CONFIG['qa_model'])
qa     = AutoModelForQuestionAnswering.from_pretrained(CONFIG['qa_model']).to(DEV).eval()

@torch.no_grad()
def cas_passage_scores(question, ps, max_len=384, topn=20):
    '''Returns (answerability_standard, answerability_softmax_proxy) per passage.'''
    enc = qa_tok([question]*len(ps), list(ps), return_tensors='pt',
                 truncation=True, max_length=max_len, padding=True)
    out = qa(**{k: v.to(DEV) for k, v in enc.items()})
    sl = out.start_logits.detach().cpu().numpy()
    el = out.end_logits.detach().cpu().numpy()

    std, soft = [], []
    for k in range(len(ps)):
        seq = enc.sequence_ids(k)
        ctx = np.array([j for j, s in enumerate(seq) if s == 1])   # passage tokens only
        null = float(sl[k, 0] + el[k, 0])

        if len(ctx) == 0:
            std.append(-1e4); soft.append(0.0); continue

        s_ctx, e_ctx = sl[k, ctx], el[k, ctx]
        best = -1e9
        for a in np.argsort(-s_ctx)[:topn]:
            for b in np.argsort(-e_ctx)[:topn]:
                if b < a or (b - a + 1) > CONFIG['max_ans_len']: continue
                best = max(best, float(s_ctx[a] + e_ctx[b]))
        if best <= -1e8: best = null - 1.0
        std.append(-(null - best))        # higher = more answerable

        # the v4 proxy, kept for comparison
        def sm0(v):
            v = v - v.max(); e = np.exp(v); return float(e[0] / e.sum())
        soft.append(1.0 - math.sqrt(sm0(sl[k]) * sm0(el[k])))

    return np.array(std), np.array(soft)

cas_max, cas_mean, cas_soft_max = [], [], []
for ps in tqdm(df['ret_text'], total=len(df), desc='CAS'):
    ps = [p for p in ps if str(p).strip()]
    a_std, a_soft = cas_passage_scores(df['question'].iloc[len(cas_max)], ps)
    cas_max.append(float(a_std.max()))
    cas_mean.append(float(a_std.mean()))
    cas_soft_max.append(float(a_soft.max()))

df['cas']          = cas_max        # the spec's max-pooled CAS, standard score
df['cas_mean']     = cas_mean       # mean-pooled: less prone to one false positive
df['cas_soft']     = cas_soft_max   # v4's definition, for the comparison
print(df[['cas','cas_mean','cas_soft']].describe().round(3).to_string())
print("\nmax-pooling fires if ANY of 5 passages looks answerable, so it inflates")
print("false positives. Report cas_mean alongside it.")

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

CAS:   0%|          | 0/300 [00:00<?, ?it/s]

           cas  cas_mean  cas_soft
count  300.000   300.000   300.000
mean     3.334    -4.904     0.692
std      6.418     4.272     0.353
min    -15.314   -16.211     0.002
25%     -1.420    -7.509     0.373
50%      3.677    -5.106     0.882
75%      9.022    -1.675     0.990
max     15.493     5.666     1.000

max-pooling fires if ANY of 5 passages looks answerable, so it inflates
false positives. Report cas_mean alongside it.


## Cell 8 — Targets and scope flags

Three targets, and they are not equally trustworthy:

* **`y_cov`** — a gold answer string appears in the passages. This is the label
  your own 21-row audit showed is wrong on **57%** of its positives. Treat every
  number computed against it as provisional.
* **`y_gen`** — flan-t5 reads the passages and its answer matches a gold answer.
* **`y_judge`** — flan-t5 is asked whether the passages support the gold answer.
  **The prompt below is not a validated instrument.** Replace it with the
  published autorater prompt from Joren et al. (2025) before reporting anything
  from it.

`evaluable` marks the 136 questions whose gold page the scan actually reached.
For the other 164 the coverage label means "we never looked there".

In [ ]:
from transformers import AutoModelForSeq2SeqLM
import torch.nn.functional as F
import unicodedata

lm_tok = AutoTokenizer.from_pretrained(CONFIG['lm'])
lm = AutoModelForSeq2SeqLM.from_pretrained(CONFIG['lm']).to(DEV).eval()

@torch.no_grad()
def seq_logprob(prompt, target):
    e = lm_tok(prompt, return_tensors='pt', truncation=True, max_length=512).to(DEV)
    l = lm_tok(target, return_tensors='pt', truncation=True, max_length=32).input_ids.to(DEV)
    lg = lm(**e, labels=l).logits
    return float(F.log_softmax(lg, -1).gather(-1, l.unsqueeze(-1)).squeeze(-1).mean())

@torch.no_grad()
def gen(prompt, n=24):
    e = lm_tok(prompt, return_tensors='pt', truncation=True, max_length=512).to(DEV)
    return lm_tok.decode(lm.generate(**e, max_new_tokens=n, num_beams=1, do_sample=False)[0],
                         skip_special_tokens=True).strip()

P_CTX = "Answer the question using only the passages.\n\nPassages:\n{ctx}\n\nQuestion: {q}\nAnswer:"
J_PLACEHOLDER = ("Passages:\n{ctx}\n\nQuestion: {q}\nProposed answer: {a}\n\n"
                 "Do the passages state or directly imply that this is the answer? "
                 "Answer yes or no.\nAnswer:")

def norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii','ignore').decode().lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s); s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return ' '.join(s.split())

def hit(pred, golds, short_len=4):
    '''PopQA ships country codes (us, cn, ba) as aliases, so plain substring
    matching makes "us" match inside "business". Short golds need word boundaries.'''
    p = norm(pred)
    if not p: return 0
    for g in golds:
        g = norm(g)
        if not g: continue
        if len(g) <= short_len:
            if re.search(rf'(?<![a-z0-9]){re.escape(g)}(?![a-z0-9])', p): return 1
        elif g == p or g in p or p in g: return 1
    return 0

_T = [("the United States",["United States of America","USA","US"],1),
      ("Praha",["Prague","Praha"],1),
      ("business australia",["US","United States"],0), ("discuss",["US"],0),
      ("it is in the US",["US"],1), ("cnn news",["CN","China"],0),
      ("Paris",["Leeuwarden"],0), ("",["Leeuwarden"],0)]
assert not [t for t in _T if hit(t[0], t[1]) != t[2]], "alias matcher regression"
print(f"alias matcher: {len(_T)}/{len(_T)} ok")

import ast
def parse_answers(x):
    if isinstance(x, str) and x.strip().startswith('['):
        try: return [str(a) for a in ast.literal_eval(x)]
        except Exception: pass
    return [s.strip() for s in str(x).split('|') if s.strip()]
df['answers'] = df['answers_list'].apply(parse_answers)

gens, judges = [], []
for _, r in tqdm(df.iterrows(), total=len(df), desc='targets'):
    ctx  = "\n\n".join(f"[{i+1}] {p}" for i, p in enumerate(r['ret_text']) if str(p).strip())
    gold = r['answers'][0] if r['answers'] else ''
    gens.append(gen(P_CTX.format(ctx=ctx, q=r['question'])))
    pr = J_PLACEHOLDER.format(ctx=ctx, q=r['question'], a=gold)
    judges.append(int(seq_logprob(pr, 'yes') > seq_logprob(pr, 'no')))

df['gen_ctx']   = gens
df['y_gen']     = [hit(p, g) for p, g in zip(df['gen_ctx'], df['answers'])]
df['y_judge']   = judges                       # PLACEHOLDER — not validated
df['y_cov']     = df['m3_covered'].astype(int)
df['is_tail']   = (df['band'].astype(str).str.lower() == 'tail').astype(int)

# use the full-pass M2 if the scan notebook has been run, else the old partial one
M2COL = 'm2_status_v5' if 'm2_status_v5' in df.columns else 'm2_status'
df['evaluable'] = df[M2COL].isin(['found','not_found'])
print(f"evaluable flag from: {M2COL}")

for t in ['y_cov','y_gen','y_judge']:
    print(f"  {t:<8} positive {int(df[t].sum()):>3}/{len(df)}  ({df[t].mean():.3f})")
print(f"  evaluable (coverage actually known): {int(df['evaluable'].sum())}/{len(df)}")
df.drop(columns=['ret_text']).to_csv(CONFIG['out_signals'], index=False)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

alias matcher: 8/8 ok


targets:   0%|          | 0/300 [00:00<?, ?it/s]

evaluable flag from: m2_status_v5
  y_cov    positive 123/300  (0.410)
  y_gen    positive 106/300  (0.353)
  y_judge  positive 124/300  (0.413)
  evaluable (coverage actually known): 134/300


## Cell 9 — Validate CAS against your 50 hand labels

This is the only human ground truth in the project. `SUPPORTED` vs
`NOT_SUPPORTED`, written by you, with reasons. `UNCERTAIN` rows are dropped.

If CAS separates these well, you have a validated signal and a citable sentence.
If it does not, you have learned that for the price of 50 rows instead of at the
end of the project. Either outcome is worth having.

In [ ]:
from sklearn.metrics import roc_auc_score

if not os.path.exists(CONFIG['labelled_csv']):
    print("hand-labelled CSV not found — skipping. Put halt_rag_audit_prelabeled.csv in Drive.")
else:
    lab = pd.read_csv(CONFIG['labelled_csv'])[['question','evidence_label','priority_group']]
    m = df.merge(lab, on='question', how='inner')
    m = m[m['evidence_label'].isin(['SUPPORTED','NOT_SUPPORTED'])].copy()
    m['y_human'] = (m['evidence_label'] == 'SUPPORTED').astype(int)
    print(f"matched {len(m)} labelled rows "
          f"({int(m.y_human.sum())} SUPPORTED / {int((1-m.y_human).sum())} NOT_SUPPORTED)")

    if m['y_human'].nunique() < 2:
        print("only one class — cannot score.")
    else:
        print("\nAUROC against YOUR hand labels (the honest test):")
        for s in ['cas','cas_mean','cas_soft','lift','relevance','consistency',
                  'kw_coverage','clarity','y_cov']:
            if s in m.columns:
                a = roc_auc_score(m['y_human'], m[s])
                flag = ''
                if s == 'y_cov': flag = '   <- the string match itself'
                print(f"  {s:<14} {a:.4f}  (flipped {max(a,1-a):.4f}){flag}")
        print("\nNote n is small, so treat these as a sanity check, not a result.")
        print("The y_cov row tells you how good your automatic label is against your own eyes.")

## Cell 10 — Per-signal results, with the leak check, on both scopes

`BAND?` means the signal predicts head/tail better than it predicts the target.
Such a signal is detecting entity popularity, not retrieval quality.
`noise_ctrl` is random numbers and must sit near 0.50.

In [ ]:
from sklearn.metrics import average_precision_score

SIGNALS = ['lift','relevance','consistency','kw_coverage','clarity','cas']
EXTRA   = ['cas_mean','cas_soft']
df['noise_ctrl'] = np.random.default_rng(CONFIG['seed']).normal(size=len(df))
EVAL = SIGNALS + EXTRA + ['noise_ctrl']

SCOPES = {'all_300': df, 'evaluable_136': df[df['evaluable']]}
rows = []
for scope, sub in SCOPES.items():
    it = (sub['is_tail'] == 1).astype(int).values
    for tgt in ['y_cov','y_gen','y_judge']:
        y = sub[tgt].values
        if len(np.unique(y)) < 2:
            print(f"[{scope}/{tgt}] single class — skipped"); continue
        for s in EVAL:
            v = sub[s].values
            a  = roc_auc_score(y, v)
            ab = roc_auc_score(it, v) if len(np.unique(it)) > 1 else 0.5
            rows.append(dict(scope=scope, target=tgt, signal=s, n=len(sub),
                             auroc=a, auroc_flipped=max(a, 1-a),
                             auprc=average_precision_score(y, v),
                             band_auroc=max(ab, 1-ab),
                             leak='BAND?' if max(ab,1-ab) > max(a,1-a) else ''))
res = pd.DataFrame(rows)

for scope in SCOPES:
    for tgt in ['y_cov','y_gen','y_judge']:
        t = res[(res.scope==scope) & (res.target==tgt)]
        if not len(t): continue
        base = SCOPES[scope][tgt].mean()
        print(f"\n=== {scope} / {tgt}  (n={len(SCOPES[scope])}, base rate {base:.3f}) ===")
        print(t.sort_values('auroc_flipped', ascending=False)
               [['signal','auroc','auroc_flipped','auprc','band_auroc','leak']]
               .round(3).to_string(index=False))

for scope, sub in SCOPES.items():
    it = (sub['is_tail']==1).astype(int).values
    print(f"\ncorr(y_cov, is_tail) [{scope}] = {np.corrcoef(sub['y_cov'], it)[0,1]:+.4f}")

## Cell 11 — SWL and the ablation

Logistic regression over the signals, scored out of fold. This matches what the
literature review claims and keeps one interpretable weight per signal, which is
what RQ2 is actually asking about.

Pick the scope deliberately. `evaluable_136` is the honest one but has far fewer
negative cases, so the numbers will be noisy — the cell prints the count so you
can judge it.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

SCOPE  = 'all_300'      # <- switch to 'evaluable_136'
TARGET = 'y_cov'        # <- switch to 'y_gen' once you trust it

work = SCOPES[SCOPE]
y = work[TARGET].values
print(f"scope={SCOPE}  target={TARGET}  n={len(work)}  "
      f"positives={int(y.sum())}  negatives={int((1-y).sum())}")
if min(int(y.sum()), int((1-y).sum())) < 30:
    print("!! fewer than 30 cases in the smaller class — treat all of this as indicative only.")

def swl(X, y, seed=CONFIG['seed']):
    if len(np.unique(y)) < 2:
        raise ValueError("single-class target — this is what broke the earliest run")
    p = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=seed))
    oof = cross_val_predict(p, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=seed),
                            method='predict_proba')[:, 1]
    p.fit(X, y)
    return oof, p.named_steps['logisticregression'].coef_[0]

CHEAP = ['lift','relevance','consistency','kw_coverage','clarity']
ALL6  = CHEAP + ['cas']
CONFIGS = {
    'SWL-A  Lift only'            : ['lift'],
    'SWL-B  5 cheap signals'      : CHEAP,
    'SWL-C  6 signals (+CAS)'     : ALL6,
    'SWL-C2 6 signals (cas_mean)' : CHEAP + ['cas_mean'],
    'kw_coverage alone'           : ['kw_coverage'],
    'cas alone'                   : ['cas'],
}
out = []
for name, feats in CONFIGS.items():
    oof, _ = swl(work[feats].values, y)
    out.append(dict(config=name, k=len(feats), auroc=roc_auc_score(y, oof)))

full_oof, W = swl(work[ALL6].values, y)
full = roc_auc_score(y, full_oof)
for s in ALL6:
    keep = [f for f in ALL6 if f != s]
    out.append(dict(config=f'  minus {s}', k=len(keep),
                    auroc=roc_auc_score(y, swl(work[keep].values, y)[0])))

# FIX 7 — this baseline is only meaningful when the target is NOT the string match
if TARGET != 'y_cov':
    out.append(dict(config='string-match baseline', k=1,
                    auroc=roc_auc_score(y, (work['answer_string_match']=='YES').astype(int))))
else:
    print("\n(string-match baseline skipped: y_cov IS the string match, so it would score 1.000)")

o = pd.DataFrame(out); o['delta_vs_6sig'] = (o['auroc'] - full).round(3)
print(); print(o.round(3).to_string(index=False))
print(f"\n6-signal weights: {dict(zip(ALL6, W.round(3)))}")
print("\nIf SWL-C is not clearly above the best single signal, combining buys nothing.")
print("That is a legitimate negative answer to RQ2 — report it.")

## Cell 12 — Abstention operating points

In [ ]:
pts = []
for cov in [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]:
    t   = np.quantile(full_oof, 1 - cov)
    ans = full_oof >= t
    pts.append(dict(coverage=round(float(ans.mean()), 3),
                    acc_on_answered=round(float(y[ans].mean()), 3) if ans.any() else None,
                    abstention_precision=round(float((1-y[~ans]).mean()), 3) if (~ans).any() else None,
                    n_abstained=int((~ans).sum())))
pts = pd.DataFrame(pts)
print(f"answer everything -> accuracy {y.mean():.3f}\n")
print(pts.to_string(index=False))

## Cell 13 — Save

In [ ]:
df.drop(columns=['ret_text']).to_csv(CONFIG['out_signals'], index=False)
with open(CONFIG['out_results'], 'w') as f:
    json.dump(dict(config=CONFIG, scope=SCOPE, target=TARGET, n=len(work),
                   corpus_passages=len(passages),
                   base_rates={t: float(df[t].mean()) for t in ['y_cov','y_gen','y_judge']},
                   n_evaluable=int(df['evaluable'].sum()),
                   per_signal=res.to_dict('records'),
                   configs=o.to_dict('records'),
                   swl_weights=dict(zip(ALL6, map(float, W))),
                   operating_points=pts.to_dict('records')), f, indent=2, default=str)
print("saved:", CONFIG['out_signals'])
print("saved:", CONFIG['out_results'])
print("\nIMPORTANT: File > Download > .ipynb WITH OUTPUTS, or these numbers are gone.")
print("None of your eleven earlier notebooks saved its outputs.")

---

### Still outstanding after this notebook runs

1. **`y_cov` is still the 57%-wrong string match.** Everything measured against
   it is provisional. The real fix is relabelling all 300 rows with the published
   Joren et al. autorater prompt, which also solves the missing second annotator.
2. **`y_judge` uses an invented prompt.** Do not report it until that prompt is
   replaced with the published one.
3. **The corpus is still a pilot**, even with more distractors. The scan reached
   only 120 of your 300 entity pages.
4. **The three-way router (RQ4) is not built.** Your 14 `A_retrieval_failure`
   rows are the ground truth for it.